In [1]:
import pandas as pd
import numpy as np
import warnings
import os

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
# Load raw training data
app_train = pd.read_csv('../data/application_train.csv')

print(f"Dataset loaded: {app_train.shape[0]:,} rows, {app_train.shape[1]} columns")
print(f"Target distribution:\n{app_train['TARGET'].value_counts()}")

Dataset loaded: 307,511 rows, 122 columns
Target distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64


In [3]:
# Separate target variable before any transformations
TARGET = app_train['TARGET']
app_train = app_train.drop(columns=['TARGET'])

# Check for duplicates
duplicates = app_train.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

# Drop SK_ID_CURR — it's an identifier, not a feature
app_train = app_train.drop(columns=['SK_ID_CURR'])

print(f"Working dataset shape: {app_train.shape}")
print(f"Target shape: {TARGET.shape}")

Duplicate rows: 0
Working dataset shape: (307511, 120)
Target shape: (307511,)


In [4]:
# Fix DAYS_EMPLOYED anomaly before any feature engineering
# 365243 is a placeholder value for pensioners/unemployed — not a real employment duration
app_train['DAYS_EMPLOYED_ANOMALY'] = (app_train['DAYS_EMPLOYED'] == 365243).astype(int)
app_train['DAYS_EMPLOYED'] = app_train['DAYS_EMPLOYED'].replace(365243, np.nan)

print(f"DAYS_EMPLOYED anomaly flag created")
print(f"Anomaly cases flagged: {app_train['DAYS_EMPLOYED_ANOMALY'].sum():,}")
print(f"DAYS_EMPLOYED missing after fix: {app_train['DAYS_EMPLOYED'].isnull().sum():,}")

DAYS_EMPLOYED anomaly flag created
Anomaly cases flagged: 55,374
DAYS_EMPLOYED missing after fix: 55,374


In [5]:
# Create all new features identified during EDA
# Age in years (DAYS_BIRTH is negative — days before application)
app_train['AGE_YEARS'] = abs(app_train['DAYS_BIRTH']) / 365

# Employment in years (anomaly already handled in Cell 4)
app_train['EMPLOYMENT_YEARS'] = abs(app_train['DAYS_EMPLOYED']) / 365

# External source composite scores
app_train['EXT_SOURCE_MEAN'] = app_train[['EXT_SOURCE_1', 
                                           'EXT_SOURCE_2', 
                                           'EXT_SOURCE_3']].mean(axis=1)

app_train['EXT_SOURCE_MIN'] = app_train[['EXT_SOURCE_1', 
                                          'EXT_SOURCE_2', 
                                          'EXT_SOURCE_3']].min(axis=1)

# Car ownership flag (missingness = no car)
app_train['HAS_CAR'] = app_train['OWN_CAR_AGE'].notnull().astype(int)

# Financial ratios
app_train['CREDIT_INCOME_RATIO'] = (app_train['AMT_CREDIT'] / 
                                     app_train['AMT_INCOME_TOTAL'])

app_train['ANNUITY_INCOME_RATIO'] = (app_train['AMT_ANNUITY'] / 
                                      app_train['AMT_INCOME_TOTAL'])

app_train['LOAN_TERM'] = (app_train['AMT_CREDIT'] / 
                           app_train['AMT_ANNUITY'])

app_train['INCOME_PER_PERSON'] = (app_train['AMT_INCOME_TOTAL'] / 
                                   app_train['CNT_FAM_MEMBERS'])

# Winsorise extreme outliers at 99th percentile
cols_to_winsorise = [
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 
    'AMT_ANNUITY', 'AMT_GOODS_PRICE'
]

for col in cols_to_winsorise:
    cap = app_train[col].quantile(0.99)
    app_train[col] = app_train[col].clip(upper=cap)
    print(f"{col} capped at: {cap:,.0f}")

print(f"\nNew features created successfully")
print(f"Dataset shape after engineering: {app_train.shape}")

AMT_INCOME_TOTAL capped at: 472,500
AMT_CREDIT capped at: 1,854,000
AMT_ANNUITY capped at: 70,006
AMT_GOODS_PRICE capped at: 1,800,000

New features created successfully
Dataset shape after engineering: (307511, 130)


In [6]:
# Separate columns by type — the pipeline treats them differently
categorical_cols = app_train.select_dtypes(include=['object']).columns.tolist()
numerical_cols = app_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical columns: {len(categorical_cols)}")
print(f"Numerical columns: {len(numerical_cols)}")
print(f"\nCategorical columns:")
for col in categorical_cols:
    n_unique = app_train[col].nunique()
    n_missing = app_train[col].isnull().sum()
    print(f"  {col}: {n_unique} unique values, {n_missing:,} missing")

Categorical columns: 16
Numerical columns: 114

Categorical columns:
  NAME_CONTRACT_TYPE: 2 unique values, 0 missing
  CODE_GENDER: 3 unique values, 0 missing
  FLAG_OWN_CAR: 2 unique values, 0 missing
  FLAG_OWN_REALTY: 2 unique values, 0 missing
  NAME_TYPE_SUITE: 7 unique values, 1,292 missing
  NAME_INCOME_TYPE: 8 unique values, 0 missing
  NAME_EDUCATION_TYPE: 5 unique values, 0 missing
  NAME_FAMILY_STATUS: 6 unique values, 0 missing
  NAME_HOUSING_TYPE: 6 unique values, 0 missing
  OCCUPATION_TYPE: 18 unique values, 96,391 missing
  WEEKDAY_APPR_PROCESS_START: 7 unique values, 0 missing
  ORGANIZATION_TYPE: 58 unique values, 0 missing
  FONDKAPREMONT_MODE: 4 unique values, 210,295 missing
  HOUSETYPE_MODE: 3 unique values, 154,297 missing
  WALLSMATERIAL_MODE: 7 unique values, 156,341 missing
  EMERGENCYSTATE_MODE: 2 unique values, 145,755 missing


In [7]:
# Drop columns where missingness is so high they cannot be reliably imputed
# and where the data is structurally tied to property type (low predictive value)
cols_to_drop = [
    'FONDKAPREMONT_MODE',    # 68% missing — property maintenance fund
    'COMMONAREA_AVG',        # 70% missing — building common area
    'COMMONAREA_MODE',       # 70% missing
    'COMMONAREA_MEDI',       # 70% missing
    'NONLIVINGAPARTMENTS_AVG',  # 69% missing
    'NONLIVINGAPARTMENTS_MODE', # 69% missing
    'NONLIVINGAPARTMENTS_MEDI', # 69% missing
    'LIVINGAPARTMENTS_AVG',  # 68% missing
    'LIVINGAPARTMENTS_MODE', # 68% missing
    'LIVINGAPARTMENTS_MEDI', # 68% missing
    'FLOORSMIN_AVG',         # 68% missing
    'FLOORSMIN_MODE',        # 68% missing
    'FLOORSMIN_MEDI',        # 68% missing
    'YEARS_BUILD_AVG',       # 66% missing
    'YEARS_BUILD_MODE',      # 66% missing
    'YEARS_BUILD_MEDI',      # 66% missing
    'AMT_GOODS_PRICE',       # 99% correlated with AMT_CREDIT — redundant
    'CNT_FAM_MEMBERS',       # 88% correlated with CNT_CHILDREN — redundant
]

# Only drop columns that exist in the dataset
cols_to_drop = [col for col in cols_to_drop if col in app_train.columns]

app_train = app_train.drop(columns=cols_to_drop)

# Update column lists
categorical_cols = app_train.select_dtypes(include=['object']).columns.tolist()
numerical_cols = app_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Dropped {len(cols_to_drop)} columns")
print(f"Remaining categorical columns: {len(categorical_cols)}")
print(f"Remaining numerical columns: {len(numerical_cols)}")
print(f"Final dataset shape: {app_train.shape}")

Dropped 18 columns
Remaining categorical columns: 15
Remaining numerical columns: 97
Final dataset shape: (307511, 112)


In [8]:
# Define numerical and categorical pipelines separately
# then combine them using ColumnTransformer

numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine into a single ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('numerical', numerical_pipeline, numerical_cols),
    ('categorical', categorical_pipeline, categorical_cols)
])

print("Pipeline defined successfully")
print(f"\nNumerical pipeline steps:")
for step in numerical_pipeline.steps:
    print(f"  {step[0]}: {step[1]}")
print(f"\nCategorical pipeline steps:")
for step in categorical_pipeline.steps:
    print(f"  {step[0]}: {step[1]}")

Pipeline defined successfully

Numerical pipeline steps:
  imputer: SimpleImputer(strategy='median')
  scaler: StandardScaler()

Categorical pipeline steps:
  imputer: SimpleImputer(strategy='most_frequent')
  encoder: OneHotEncoder(handle_unknown='ignore', sparse_output=False)


In [9]:
# Fit the pipeline on the training data and transform it
print("Fitting and transforming data — this may take a minute...")

X_processed = preprocessor.fit_transform(app_train)

# Get feature names after transformation
numerical_feature_names = numerical_cols

categorical_feature_names = preprocessor.named_transformers_['categorical']\
    ['encoder'].get_feature_names_out(categorical_cols).tolist()

all_feature_names = numerical_feature_names + categorical_feature_names

# Convert to DataFrame
X_processed_df = pd.DataFrame(X_processed, columns=all_feature_names)

print(f"Transformation complete")
print(f"Processed dataset shape: {X_processed_df.shape}")
print(f"Numerical features: {len(numerical_feature_names)}")
print(f"Categorical features after encoding: {len(categorical_feature_names)}")
print(f"\nMissing values remaining: {X_processed_df.isnull().sum().sum()}")

Fitting and transforming data — this may take a minute...
Transformation complete
Processed dataset shape: (307511, 233)
Numerical features: 97
Categorical features after encoding: 136

Missing values remaining: 0


In [10]:
import os

# Save processed features and target variable
output_dir = '../data'
os.makedirs(output_dir, exist_ok=True)

X_processed_df.to_csv(f'{output_dir}/X_processed.csv', index=False)
TARGET.to_csv(f'{output_dir}/y_target.csv', index=False)

print(f"Processed features saved: data/X_processed.csv")
print(f"Target variable saved: data/y_target.csv")
print(f"\n=== STEP 3 COMPLETE ===")
print(f"Input:  307,511 rows x 120 columns (raw)")
print(f"Output: 307,511 rows x 233 columns (processed)")
print(f"Missing values: 0")
print(f"Pipeline: fitted and ready for deployment")

Processed features saved: data/X_processed.csv
Target variable saved: data/y_target.csv

=== STEP 3 COMPLETE ===
Input:  307,511 rows x 120 columns (raw)
Output: 307,511 rows x 233 columns (processed)
Missing values: 0
Pipeline: fitted and ready for deployment


In [11]:
import joblib
import os

os.makedirs('../src', exist_ok=True)

joblib.dump(preprocessor, '../src/preprocessor.pkl')

print("Pipeline saved to src/preprocessor.pkl")
print("\nThis file is critical for deployment.")
print("It ensures new applicant data is transformed")
print("identically to the training data.")

Pipeline saved to src/preprocessor.pkl

This file is critical for deployment.
It ensures new applicant data is transformed
identically to the training data.
